# Daily Brief AI — Kaggle version

**Before running (one-time setup):**
1. **Notebook Settings (right sidebar) → Internet: ON.** Off by default; NewsData API, RSS feeds, article scraping, and email will all fail silently without this.
2. **Add-ons → Secrets** — add these four, matching names exactly:
   - `NEWSDATA_API_KEY`
   - `GMAIL_ADDRESS`
   - `GMAIL_APP_PASSWORD`
   - `RECIPIENT_EMAIL`
3. To automate: Settings → **Schedule a notebook to run** → Daily. (Scheduled notebooks are **CPU-only** — GPU cannot be combined with scheduling on Kaggle, so Ollama will run on CPU here. The 1.5B/1.7B models are workable; the 7B selector call will be noticeably slower than on a GPU.)

Everything below is otherwise **identical** to the working Colab version — same agents, same tasks, same prompts, same retry/fallback logic. Only environment glue (secrets, file paths) changed.

In [1]:
!pip install crewai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 7.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.4/202.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 33.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 35.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━

In [2]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 227 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (1,710 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%   12.7%#################                              61.9%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess
import time

# Start Ollama server in background without shell errors
process = subprocess.Popen(["ollama", "serve"])

# Pause 5 seconds for Ollama service startup
time.sleep(5)
print("[✓] Ollama background server started cleanly!")

Couldn't find '/root/.ollama/id_ed25519'. Generating new private key.
Your new public key is: 

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIJX0KAr7djrB9LdOeL1MAaa3/ga3+Q0vL8fYTMXziEAw



time=2026-09-17T17:17:39.059Z level=INFO source=routes.go:1941 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https:

[✓] Ollama background server started cleanly!


In [5]:
!pip install -q crewai requests python-dotenv beautifulsoup4 reportlab

time=2026-09-17T17:17:47.099Z level=INFO source=types.go:32 msg="inference compute" id=0 filter_id=0 library=CUDA compute=7.5 name=CUDA0 description="Tesla T4" libdirs=ollama,cuda_v13 driver=13.0 pci_id=0000:00:04.0 type=discrete total="14.6 GiB" available="14.5 GiB"
time=2026-09-17T17:17:47.099Z level=INFO source=types.go:32 msg="inference compute" id=1 filter_id=1 library=CUDA compute=7.5 name=CUDA1 description="Tesla T4" libdirs=ollama,cuda_v13 driver=13.0 pci_id=0000:00:05.0 type=discrete total="14.6 GiB" available="14.5 GiB"
time=2026-09-17T17:17:47.099Z level=INFO source=routes.go:2051 msg="vram-based default context" total_vram="29.1 GiB" default_num_ctx=32768


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 24.1 MB/s eta 0:00:0000:0100:01


In [6]:
!ollama pull qwen2.5:7b-instruct


]11;?\[GIN] 2026/09/17 - 17:17:56 | 200 |      89.076µs |       127.0.0.1 | HEAD     "/"
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ 

time=2026-09-17T17:17:57.517Z level=INFO source=download.go:181 msg="downloading 2bada8a74506 in 16 292 MB part(s)"


pulling manifest ⠧ pulling manifest 
pulling 2bada8a74506:   0% ▕                  ▏ 4.3 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   1% ▕                  ▏  34 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   2% ▕                  ▏ 116 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   4% ▕                  ▏ 202 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   5% ▕                  ▏ 241 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   7% ▕█                 ▏ 332 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:   9% ▕█                 ▏ 421 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  10% ▕█                 ▏ 460 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  12% ▕██                ▏ 550 MB/4.7 GB                  pulling manifest 
pulling 2bada8a74506:  14% ▕██                ▏ 634 MB/4.7 GB                  pulling man

time=2026-09-17T17:18:09.775Z level=INFO source=download.go:181 msg="downloading 66b9ea09bd5b in 1 68 B part(s)"


pulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0spulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0spulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0s

...

pulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0spulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0spulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0s

time=2026-09-17T17:18:10.413Z level=INFO source=download.go:181 msg="downloading 832dd9e00a68 in 1 11 KB part(s)"


pulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0spulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0spulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0s

time=2026-09-17T17:18:10.705Z level=INFO source=download.go:181 msg="downloading 2f15b3218f05 in 1 487 B part(s)"


pulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0s
verifying sha256 digest ⠋ pulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0s
verifying sha256 digest ⠙ pulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0s
verifying sha256 digest ⠹ pulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0s
verifying sha256 digest ⠸ pulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0s
verifying sha256 digest ⠼ pulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0s
verifying sha256 digest ⠴ pulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0s
verifying sha256 digest ⠦ pulling manifest 
pulling 2bada8a74506: 100% ▕█████████████████ ▏ 4.7 GB/4.7 GB  359 MB/s      0s
verifying sha256 digest ⠧ pulling 

In [8]:
!ollama pull qwen3:1.7b

]11;?\[GIN] 2026/09/17 - 17:18:45 | 200 |      31.426µs |       127.0.0.1 | HEAD     "/"
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ 

time=2026-09-17T17:18:46.001Z level=INFO source=download.go:181 msg="downloading 3d0b790534fe in 14 100 MB part(s)"


pulling manifest 
pulling 3d0b790534fe:   0% ▕                  ▏ 1.5 MB/1.4 GB                  pulling manifest 
pulling 3d0b790534fe:   1% ▕                  ▏  19 MB/1.4 GB                  pulling manifest 
pulling 3d0b790534fe:   4% ▕                  ▏  48 MB/1.4 GB                  pulling manifest 
pulling 3d0b790534fe:   9% ▕█                 ▏ 118 MB/1.4 GB                  pulling manifest 
pulling 3d0b790534fe:  15% ▕██                ▏ 201 MB/1.4 GB                  pulling manifest 
pulling 3d0b790534fe:  20% ▕███               ▏ 274 MB/1.4 GB                  pulling manifest 
pulling 3d0b790534fe:  23% ▕████              ▏ 311 MB/1.4 GB                  pulling manifest 
pulling 3d0b790534fe:  27% ▕████              ▏ 372 MB/1.4 GB                  pulling manifest 
pulling 3d0b790534fe:  32% ▕█████             ▏ 436 MB/1.4 GB                  pulling manifest 
pulling 3d0b790534fe:  34% ▕██████            ▏ 464 MB/1.4 GB                  pulling manifest 
pulling 3d0b

time=2026-09-17T17:18:49.890Z level=INFO source=download.go:181 msg="downloading ae370d884f10 in 1 1.7 KB part(s)"


pulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0spulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0spulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0s

...

pulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0spulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0spulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0s

time=2026-09-17T17:18:50.520Z level=INFO source=download.go:181 msg="downloading cff3f395ef37 in 1 120 B part(s)"


pulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0spulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0spulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0s

time=2026-09-17T17:18:50.834Z level=INFO source=download.go:181 msg="downloading 517ccaff02fe in 1 487 B part(s)"


pulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0s
verifying sha256 digest ⠋ pulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0s
verifying sha256 digest ⠙ pulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0s
verifying sha256 digest ⠹ pulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0s
verifying sha256 digest ⠸ pulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0s
verifying sha256 digest ⠼ pulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0s
verifying sha256 digest ⠴ pulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0s
verifying sha256 digest ⠦ pulling manifest 
pulling 3d0b790534fe: 100% ▕█████████████████ ▏ 1.4 GB/1.4 GB  431 MB/s      0s
verifying sha256 digest ⠧ pulling 

In [25]:
# ============================================================
# CELL 1: setup
# ============================================================
import os
import re
import json
import requests
from bs4 import BeautifulSoup
from crewai import Agent, Task, Crew, LLM
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()
NEWSDATA_API_KEY = _secrets.get_secret('NEWSDATA_API_KEY')

# Bigger model for the ONE step that does real multi-factor judgment across
# a noisy pool of candidates (selector_agent).
selector_llm = LLM(
    model="ollama/qwen2.5:7b-instruct",
    base_url="http://localhost:11434",
    temperature=0.2,
)

# Small/fast model for steps that only operate on one already-clean input
# (summarizer_agent) — no comparison or judgment needed there.
fast_llm = LLM(
    model="ollama/qwen3:1.7b",
    base_url="http://localhost:11434",
    temperature=0.2,
)

USER_PREFERRED_TOPICS = [
    "latest technology inventions",
    "politics",
    "artificial intelligence",
    "business",
    "technology",
    "economy",
    "world news",
    "AI companies",
]

ALL_TOPICS = [
    "artificial intelligence", "technology", "business", "science",
    "world news", "politics", "economy", "space",
    "cybersecurity", "geopolitics",
]

# Soft credibility signal fed to the selector — NOT a hard filter, since this
# list is short and would zero out the pool for niche topics. Expand freely.
TRUSTED_SOURCES = {
    "reuters", "associated press", "ap", "bbc", "the guardian", "bloomberg",
    "the new york times", "the washington post", "npr", "the wall street journal",
    "the hindu", "the indian express", "hindustan times", "livemint",
    "business standard", "press trust of india", "pti",
}



In [26]:

# ============================================================
# CELL 2: deterministic helpers (scraping + cleaning)
# This IS the cleaning step. Nothing downstream should touch the wording
# of the article again.
# ============================================================

def count_words(text):
    return len(text.split()) if text else 0


GARBAGE_PHRASES = [
    "please follow us", "subscribe to our newsletter", "sign up for our newsletter",
    "read more:", "similar news", "related news", "related articles",
    "write comment", "follow us on google", "you can also read", "recommended stories",
]

CONTAINER_SELECTORS = [
    '[class*="article-body"]', '[class*="article-content"]',
    '[class*="story-body"]', '[class*="story-content"]',
    '[class*="post-content"]', '[class*="entry-content"]',
    '[class*="articleBody"]', '[class*="articleContent"]',
    '[id*="article-body"]', '[id*="article-content"]',
    '[id*="story-body"]', '[id*="story-content"]',
]


def extract_article_text(url):
    """Scrape + deterministically strip boilerplate. Output of this function
    is treated as final article text — never re-generated by an LLM."""
    try:
        response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
        if response.status_code != 200:
            return ""

        soup = BeautifulSoup(response.text, "html.parser")
        for tag in soup(["script", "style", "nav", "header", "footer", "aside",
                          "form", "noscript", "iframe", "svg"]):
            tag.decompose()

        article = soup.find("article")
        if not article:
            for selector in CONTAINER_SELECTORS:
                article = soup.select_one(selector)
                if article:
                    break

        paragraphs = article.find_all("p") if article else soup.find_all("p")

        cleaned = []
        for p in paragraphs:
            text = p.get_text(" ", strip=True)
            if not text or len(text.split()) < 5:
                continue
            if any(phrase in text.lower() for phrase in GARBAGE_PHRASES):
                continue
            cleaned.append(text)

        return "\n\n".join(cleaned)

    except Exception as e:
        print("Extraction error:", e)
        return ""

In [27]:

# ============================================================
# CELL 3: candidate collection
# Full text IS scraped (needed for an accurate word count and for the
# selector to judge sourcing/substance) but only a short excerpt — plus
# metadata — is what actually goes into the LLM prompt. Full text is
# cached locally so the winner's body doesn't need re-scraping later.
# ============================================================

EXCERPT_WORDS = 400

def collect_candidates(topic, max_articles=8, country=None):
    url = "https://newsdata.io/api/1/latest"
    params = {"apikey": NEWSDATA_API_KEY, "q": topic, "language": "en", "size": max_articles}
    if country:
        params["country"] = country  # e.g. "in" — restricts to India-origin publishers

    try:
        response = requests.get(url, params=params, timeout=20)
        data = response.json()
    except Exception as e:
        print("API error:", e)
        return []

    if data.get("status") != "success":
        print("API Error:", data)
        return []

    candidates = []
    for item in data.get("results", []):
        link = item.get("link", "")
        if not link:
            continue

        full_text = extract_article_text(link)
        wc = count_words(full_text)
        if wc < 220:
            continue

        source_name = item.get("source_name", "")
        candidates.append({
            "title": item.get("title", ""),
            "source": source_name,
            "url": link,
            "published_date": item.get("pubDate", ""),
            "word_count": wc,
            "known_reliable_source": source_name.strip().lower() in TRUSTED_SOURCES,
            "excerpt": " ".join(full_text.split()[:EXCERPT_WORDS]),
            "_full_text": full_text,  # cached, never sent to the LLM directly
        })

    return candidates


def build_candidate_pool(topics, per_topic=6, max_total=20, include_india=True):
    pool = {}
    for topic in topics:
        for c in collect_candidates(topic, per_topic):
            pool[c["url"]] = c  # dedupe across topics by URL
        if include_india:
            # same topic, scoped to India-origin publishers — additive, not
            # a replacement for the global query above
            for c in collect_candidates(topic, per_topic, country="in"):
                pool[c["url"]] = c

    candidates = list(pool.values())
    # keep the prompt size sane for a small model — bias the cap toward
    # longer / known-reliable candidates rather than a plain slice
    candidates.sort(key=lambda c: (c["known_reliable_source"], c["word_count"]), reverse=True)
    return candidates[:max_total]


def list_indian_sources():
    """One-off lookup: run this to get the exact domain IDs NewsData uses
    for specific Indian mastheads (The Hindu, Indian Express, etc.), if you
    later want to target them directly via the `domain` param instead of
    just `country`."""
    url = "https://newsdata.io/api/1/sources"
    params = {"apikey": NEWSDATA_API_KEY, "country": "in", "language": "en"}
    response = requests.get(url, params=params, timeout=20)
    data = response.json()
    for s in data.get("results", []):
        print(s.get("id"), "|", s.get("name"), "|", s.get("url"))




def to_selector_payload(pool):
    """Strip cached full text before this goes into a prompt."""
    return [{k: v for k, v in c.items() if k != "_full_text"} for c in pool]


In [28]:
# ============================================================
# CELL 3b: editorial/opinion candidate collection via direct RSS
# Bypasses NewsData entirely for this pool — NewsData indexes wire news,
# not analysis/opinion sections, so we go straight to the source.
# ============================================================

!pip -q install feedparser
import feedparser

EDITORIAL_FEEDS = {
    "The Hindu": "https://www.thehindu.com/opinion/editorial/feeder/default.rss",
    "The Hindu (Op-Ed)": "https://www.thehindu.com/opinion/op-ed/feeder/default.rss",
    "Indian Express (Editorials)": "https://indianexpress.com/section/opinion/editorials/feed/",
    "Indian Express (Columns)": "https://indianexpress.com/section/opinion/columns/feed/",
    "Hindustan Times (Editorials)": "https://www.hindustantimes.com/feeds/rss/editorials/rssfeed.xml",
}

def collect_editorial_candidates(max_per_feed=5):
    candidates = []
    for source_name, feed_url in EDITORIAL_FEEDS.items():
        parsed = feedparser.parse(feed_url)
        for entry in parsed.entries[:max_per_feed]:
            link = entry.get("link", "")
            if not link:
                continue
            full_text = extract_article_text(link)  # reuses your Cell 2 cleaner as-is
            wc = count_words(full_text)
            if wc < 220:
                continue
            candidates.append({
                "title": entry.get("title", ""),
                "source": source_name,
                "url": link,
                "published_date": entry.get("published", ""),
                "word_count": wc,
                "known_reliable_source": True,  # these ARE your trusted mastheads
                "excerpt": " ".join(full_text.split()[:EXCERPT_WORDS]),
                "_full_text": full_text,
            })
    return candidates

In [29]:

# ============================================================
# CELL 4: robust JSON parsing
# Small/local models often wrap "JSON only" output in ```json fences or
# add a stray sentence despite instructions. Parse defensively.
# ============================================================

def parse_json_output(raw_result):
    text = str(raw_result).strip()
    text = re.sub(r"^```(json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in model output:\n{text[:300]}")
    return json.loads(match.group(0))



In [30]:

# ============================================================
# CELL 5: selector agent + task
# ============================================================
selector_agent = Agent(
    role="Senior News Editor",
    goal=(
        "From the candidate articles provided, select exactly ONE that best "
        "rewards a reader's time — specifically a reader who wants to build "
        "general knowledge AND improve their English through reading. "
        "Prioritize, in order: "
        "(1) factual substance and depth over surface-level coverage, "
        "(2) credibility of the source and reporting (named sources, "
        "verifiable claims, absence of speculation presented as fact), "
        "(3) learning value — does the piece build an argument, explain "
        "'why' and not just 'what', and use precise, varied vocabulary a "
        "careful reader would learn from? A well-argued editorial and a "
        "well-reported news story can both score high here; a shallow "
        "quote-stitched wire report scores low even if the topic is important. "
        "(4) real-world importance — the article should inform the reader "
        "about something that matters, not just something novel, "
        "(5) recency, used only as a tie-breaker between articles that are "
        "otherwise comparable on 1-4. "
        "Do not favor opinion/editorial pieces over news, or news over "
        "opinion, by default — judge each piece on the criteria above, "
        "regardless of which bucket it came from."
    ),
    backstory=(
        "You have spent over twenty years as an editor at a major "
        "broadsheet, first as a reporter and then running the news desk. "
        "You've since also mentored non-native English speakers using "
        "newspaper reading as a study tool, so you instinctively notice "
        "which pieces reward close reading — an argument to follow, a "
        "sentence worth re-reading for its construction — versus which "
        "pieces are just competently-relayed facts with nothing to chew on. "
        "You have low tolerance for sensational framing and vague hedging "
        "('experts say', 'many believe') used to imply authority without "
        "providing it. Rejecting every candidate in a bad pool is a normal "
        "Tuesday for you — it's not a failure, running a weak story to "
        "fill space is."
    ),
    llm=selector_llm,
    verbose=True,
)

selector_task = Task(
    description="""
    You are given:
    - CANDIDATE_ARTICLES: {candidate_articles}
    - READER_PREFERRED_TOPICS: {preferred_topics}

    Each candidate includes an "excerpt" (not the full article), a
    "word_count" measured from the full scraped article, and a
    "known_reliable_source" boolean signal.

    Hard constraints — apply ALL of these before weighing judgment criteria:
    1. Only select from articles actually present in CANDIDATE_ARTICLES.
       Never invent or guess a title, URL, source, or date.
    2. Disqualify any candidate with word_count under 220.
    3. Disqualify any candidate matching these patterns, regardless of
       length, topic, or recency:
       - Listicle/roundup framing ("Top 10...", "X things you didn't know")
       - Clickbait headline framing (rhetorical hooks, manufactured urgency,
         "you won't believe")
       - Press-release style (no attributed named source or quote anywhere;
         marketing language like "thrilled to announce")
       - Vague sourcing used to imply authority ("experts say," "many
         believe") with zero named source in the excerpt
    4. Topic relevance (READER_PREFERRED_TOPICS) is a preference, not a
       filter: an excellent article outside the preferred topics beats a
       mediocre one inside them. Only use topic match to break a genuine
       quality tie.
    5. If every candidate is disqualified, or the entire pool is mediocre
       by your own standard as described in your backstory, return
       no_qualifying_article. Do NOT select a low-quality article just to
       satisfy "select exactly one" — forcing a pick you wouldn't run is a
       worse failure than reporting that nothing qualified.

    In your reasoning, name the 2-3 specific factors that made the winner
    beat the runner-up (e.g. "named sources vs. anonymous quotes in
    Article B") — not a generic restatement of the criteria list.
    """,

    expected_output="""
    A single JSON object, with no text outside the JSON:

    {
      "status": "selected",
      "title": "<exact title from input>",
      "url": "<exact url from input>",
      "source": "<publication name if available>",
      "published_date": "<if available>",
      "reason": "<2-3 sentences citing specific comparative factors>"
    }

    Or, if no candidate qualifies:

    {
      "status": "no_qualifying_article",
      "reason": "<why nothing met the bar>"
    }
    """,
    agent=selector_agent,
)



In [ ]:
# ============================================================
# CELL 6: run selection with retry/fallback (Phase 7)
# Pool is scraped ONCE. Retries only relax the selector's task text and
# re-run the (cheap) LLM call — never re-scrape. Final fallback is
# deterministic (no LLM call at all) so a bad day costs almost nothing.
# ============================================================

MAX_SELECTOR_ATTEMPTS = 3

# --- build pool once ---
news_pool = build_candidate_pool(USER_PREFERRED_TOPICS, per_topic=6, max_total=15)
editorial_pool = collect_editorial_candidates(max_per_feed=5)

merged = {}
for c in news_pool + editorial_pool:
    merged[c["url"]] = c

MAX_TOTAL = 20
candidate_pool = sorted(
    merged.values(),
    key=lambda c: (c["known_reliable_source"], c["word_count"]),
    reverse=True,
)[:MAX_TOTAL]

print(f"Pool: {len(news_pool)} news + {len(editorial_pool)} editorial "
      f"→ {len(merged)} unique → {len(candidate_pool)} sent to selector")

if not candidate_pool:
    raise SystemExit("No candidates scraped at all — check NewsData API key / editorial feed URLs / network.")


# --- relaxed task builder: same criteria, progressively softer bar ---
def build_selector_task(attempt, min_word_count):
    relax_note = ""
    if attempt == 2:
        relax_note = (
            "\nNOTE: A stricter pass already found nothing. Loosen your bar — "
            "accept reasonably good candidates even if not exceptional. "
            "Topic match is NOT a filter; do not reject solely for being "
            "outside preferred topics.\n"
        )
    elif attempt >= 3:
        relax_note = (
            "\nNOTE: Two stricter passes already found nothing. This is the "
            "last attempt before an automatic fallback. Select the single "
            "best candidate available even if it's merely acceptable rather "
            "than excellent — do not return no_qualifying_article unless "
            "every single candidate is genuinely unreadable (broken text, "
            "near-empty, or pure spam).\n"
        )

    return Task(
        description=f"""
        You are given:
        - CANDIDATE_ARTICLES: {{candidate_articles}}
        - READER_PREFERRED_TOPICS: {{preferred_topics}}
        {relax_note}
        Each candidate includes an "excerpt" (not the full article), a
        "word_count" measured from the full scraped article, and a
        "known_reliable_source" boolean signal.

        Hard constraints — apply ALL of these before weighing judgment criteria:
        1. Only select from articles actually present in CANDIDATE_ARTICLES.
           Never invent or guess a title, URL, source, or date.
        2. Disqualify any candidate with word_count under {min_word_count}.
        3. Disqualify any candidate matching these patterns, regardless of
           length, topic, or recency:
           - Listicle/roundup framing ("Top 10...", "X things you didn't know")
           - Clickbait headline framing (rhetorical hooks, manufactured urgency,
             "you won't believe")
           - Press-release style (no attributed named source or quote anywhere;
             marketing language like "thrilled to announce")
           - Vague sourcing used to imply authority ("experts say," "many
             believe") with zero named source in the excerpt
        4. Topic relevance (READER_PREFERRED_TOPICS) is a preference, not a
           filter: an excellent article outside the preferred topics beats a
           mediocre one inside them. Only use topic match to break a genuine
           quality tie.
        5. If every candidate is disqualified, return no_qualifying_article.
           Do NOT select a low-quality article just to satisfy "select
           exactly one."

        In your reasoning, name the 2-3 specific factors that made the winner
        beat the runner-up — not a generic restatement of the criteria list.
        """,
        expected_output="""
        A single JSON object, with no text outside the JSON:
        {
          "status": "selected",
          "title": "<exact title from input>",
          "url": "<exact url from input>",
          "source": "<publication name if available>",
          "published_date": "<if available>",
          "reason": "<2-3 sentences citing specific comparative factors>"
        }
        Or, if no candidate qualifies:
        {
          "status": "no_qualifying_article",
          "reason": "<why nothing met the bar>"
        }
        """,
        agent=selector_agent,
    )


# --- retry loop ---
selected = None
for attempt in range(1, MAX_SELECTOR_ATTEMPTS + 1):
    min_wc = {1: 220, 2: 150, 3: 120}[attempt]
    print(f"\n--- Selector attempt {attempt}/{MAX_SELECTOR_ATTEMPTS} (min_word_count={min_wc}) ---")

    task = build_selector_task(attempt, min_wc)
    crew = Crew(agents=[selector_agent], tasks=[task], verbose=True)

    result = await crew.kickoff_async(inputs={
        "candidate_articles": json.dumps(to_selector_payload(candidate_pool), indent=2),
        "preferred_topics": json.dumps(USER_PREFERRED_TOPICS),
    })
    parsed = parse_json_output(result)

    if parsed.get("status") == "selected":
        selected = parsed
        print(f"Selected on attempt {attempt}: {parsed.get('title')}")
        break
    else:
        print(f"Attempt {attempt} rejected: {parsed.get('reason')}")

# --- deterministic fallback: no LLM call, pick best-by-signal candidate ---
if selected is None:
    print("\nAll selector attempts exhausted — falling back to deterministic pick (no LLM call).")
    fallback = max(
        candidate_pool,
        key=lambda c: (c["known_reliable_source"], c["word_count"]),
    )
    selected = {
        "status": "selected",
        "title": fallback["title"],
        "url": fallback["url"],
        "source": fallback["source"],
        "published_date": fallback["published_date"],
        "reason": "Automatic fallback — no candidate cleared the selector's quality bar across "
                  f"{MAX_SELECTOR_ATTEMPTS} attempts; picked highest word-count/known-reliable-source candidate.",
    }
    print("Fallback selected:", selected["title"])


# --- resolve full text, same as before ---
selected_meta = next(
    (c for c in candidate_pool if c["url"] == selected["url"] or c["title"] == selected["title"]),
    None,
)
if not selected_meta:
    raise SystemExit("Selected article not found in candidate pool — title/url mismatch")

full_text = selected_meta["_full_text"]
print("\nFinal:", selected_meta["title"], "|", selected_meta["source"], "|", selected_meta["word_count"], "words")
if selected["reason"].startswith("Automatic fallback"):
    print("⚠️  Note: this was a fallback pick, not a quality-vetted selection.")

Pool: 15 news + 14 editorial → 29 unique → 20 sent to selector

--- Selector attempt 1/3 (min_word_count=220) ---


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e059d5b9-8278-4cf3-8a3e-a466a0b48a72                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          You are given:                                                                                         │
│          - CANDIDATE_ARTICLES: [                                                                                │
│    {                                                                                                            │
│      "title": "Taking a holistic view of child protection",                                                     │
│      "source": "The Hindu (Op-Ed)",                                                                             │
│      "url":                                                                                                     │
│  "https://www.thehindu.com/opinion/op-ed/taking-a-holistic-view-of-child-protection/article71473865.ece",       │
│      "published_date": "Wed, 16 Sep 2026 22:08:01 +0530",                                                       │
│      "word_count": 1557,                                                                                        │
│      "known_reliable_source": true,                                                                             │
│      "excerpt": "You don\u2019t have any Active Subscription. Subscribed with another email? Logout and Login   │
│  with that one. Account subscription benefits alongside Premium Stories, Editorials, Opinions and more. Unlock  │
│  these with Subscription Need help with your subscription? Published - September 17, 2026 12:01 am IST          │
│  Patterns such as repeat missing episodes, school discontinuity, parental loss or previous helpline contact     │
│  may help trigger support before vulnerability becomes a crisis. | Photo Credit: File The story of Saroo        │
│  Brierley is an extraordinary one. Lost at the age of five after accidentally boarding a train, separated from  │
│  his family, and eventually adopted by an Australian couple, he returned to India more than two decades later   │
│  to find his biological mother. His journey, chronicled in A Long Way Home and later adapted into the film      │
│  Lion , is a story not only of resilience but also of identity, belonging, memory, and the long consequences    │
│  of childhood separation. As implementation of the Juvenile Justice (Care and Protection of Children) Act,      │
│  2015 has completed a decade, India\u2019s child-protection system is ready for its next transition: from       │
│  collecting data about cases to understanding whether or not state intervention actually changed a              │
│  child\u2019s life. The first decade has created a substantial institutional and digital architecture. Child    │
│  Welfare Committees, Juvenile Justice Boards, District Child Protection Units, Special Juvenile Police Units,   │
│  Child Care Institutions, and helplines generate enormous amounts of information. Digital platforms             │
│  increasingly capture data on adoption, missing children, restoration, institutional and non-institutional      │
│  care, sponsorship, foster care, and aftercare. The problem, therefore, is no longer simply the absence of      │
│  data. There are data, portals, and institutional reporting systems. The more important question is, \u201cDo   │
│  we have the complete story of the child?\u201d For every child entering the protection system, data should     │
│  eventually answer seven fundamental questions: Who is the child? Why did the child become vulnerable? What     │
│  was legally required to happen? What intervention was 

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior News Editor                                                                                      │
│                                                                                                                 │
│  Task:                                                                                                          │
│          You are given:                                                                                         │
│          - CANDIDATE_ARTICLES: [                                                                                │
│    {                                                                                                            │
│      "title": "Taking a holistic view of child protection",                                                     │
│      "source": "The Hindu (Op-Ed)",                                                                             │
│      "url":                                                                                                     │
│  "https://www.thehindu.com/opinion/op-ed/taking-a-holistic-view-of-child-protection/article71473865.ece",       │
│      "published_date": "Wed, 16 Sep 2026 22:08:01 +0530",                                                       │
│      "word_count": 1557,                                                                                        │
│      "known_reliable_source": true,                                                                             │
│      "excerpt": "You don\u2019t have any Active Subscription. Subscribed with another email? Logout and Login   │
│  with that one. Account subscription benefits alongside Premium Stories, Editorials, Opinions and more. Unlock  │
│  these with Subscription Need help with your subscription? Published - September 17, 2026 12:01 am IST          │
│  Patterns such as repeat missing episodes, school discontinuity, parental loss or previous helpline contact     │
│  may help trigger support before vulnerability becomes a crisis. | Photo Credit: File The story of Saroo        │
│  Brierley is an extraordinary one. Lost at the age of five after accidentally boarding a train, separated from  │
│  his family, and eventually adopted by an Australian couple, he returned to India more than two decades later   │
│  to find his biological mother. His journey, chronicled in A Long Way Home and later adapted into the film      │
│  Lion , is a story not only of resilience but also of identity, belonging, memory, and the long consequences    │
│  of childhood separation. As implementation of the Juvenile Justice (Care and Protection of Children) Act,      │
│  2015 has completed a decade, India\u2019s child-protection system is ready for its next transition: from       │
│  collecting data about cases to understanding whether or not state intervention actually changed a              │
│  child\u2019s life. The first decade has created a substantial institutional and digital architecture. Child    │
│  Welfare Committees, Juvenile Justice Boards, District Child Protection Units, Special Juvenile Police Units,   │
│  Child Care Institutions, and helplines generate enormous amounts of information. Digital platforms             │
│  increasingly capture data on adoption, missing children, restoration, institutional and non-institutional      │
│  care, sponsorship, foster care, and aftercare. The problem, therefore, is no longer simply the absence of      │
│  data. There are data, portals, and institutional reporting systems. The more important question is, \u201cDo   │
│  we have the complete story of the child?\u201d For every child entering the protection system, data should     │
│  eventually answer seven fundamental questions: Who is 

time=2026-09-17T17:34:46.908Z level=INFO source=sched.go:1341 msg="updated VRAM based on existing loaded models" gpu=0 library=CUDA total="14.6 GiB" available="14.5 GiB"
time=2026-09-17T17:34:46.909Z level=INFO source=sched.go:1341 msg="updated VRAM based on existing loaded models" gpu=1 library=CUDA total="14.6 GiB" available="9.6 GiB"
time=2026-09-17T17:34:46.964Z level=INFO source=sched.go:1018 msg="selecting single GPU for llama-server model" main_gpu=0 id=0 filter_id=0 library=CUDA name=CUDA0 description="Tesla T4" integrated=false predicted="6.1 GiB" available="14.5 GiB"
time=2026-09-17T17:34:46.964Z level=INFO source=sched.go:1137 msg="selecting GPU backend for llama-server model" library=CUDA gpu_count=1 available_gpu_count=2
time=2026-09-17T17:34:46.964Z level=INFO source=sched.go:567 msg="llama-server model fits alongside existing models" predicted="6.9 GiB" predicted_num_ctx=32768 num_batch=1024 available="14.5 GiB" gpu_free="14.5 GiB" system_free="14.8 GiB" system_limited=f

In [17]:
!ollama list

]11;?\[GIN] 2026/09/17 - 17:22:20 | 200 |      38.501µs |       127.0.0.1 | HEAD     "/"
[GIN] 2026/09/17 - 17:22:20 | 200 |  122.185176ms |       127.0.0.1 | GET      "/api/tags"
NAME                     ID              SIZE      MODIFIED      
qwen3:1.7b               8f68893c685c    1.4 GB    3 minutes ago    
qwen2.5:1.5b-instruct    65ec06548149    986 MB    3 minutes ago    
qwen2.5:7b-instruct      845dbda0ea48    4.7 GB    3 minutes ago    


In [18]:

# ============================================================
# CELL 7: takeaways (generative — safe use of the LLM)
# The article body is NEVER regenerated by the LLM. It passes through
# untouched from the deterministic scrape/clean step above. The LLM is
# only used for something it's actually suited for: extracting takeaways.
# ============================================================

summarizer_agent = Agent(
    role="Content Summarizer",
    goal=(
        "Given a cleaned article, extract 3-5 concrete, specific takeaways "
        "a reader should walk away with. Never touch or reproduce the "
        "article body — it is already final."
    ),
    backstory=(
        "You read finished, publication-ready articles and pull out the "
        "handful of facts a busy reader needs to remember. You never "
        "invent a claim that isn't in the text, and you never pad a "
        "takeaway with vague language when a specific number or name is "
        "available in the article."
    ),
    llm=fast_llm,
    verbose=True,
)

summarizer_task = Task(
    description="""
    ARTICLE:
    {raw_article}

    Write 3-5 short, specific takeaways a reader should remember. Each
    takeaway must be a concrete fact or claim actually stated in the
    article above — not a vague restatement, and not anything invented.
    """,
    expected_output="""
    Return ONLY valid JSON:
    { "takeaways": ["...", "...", "..."] }
    """,
    agent=summarizer_agent,
)

summarizer_crew = Crew(agents=[summarizer_agent], tasks=[summarizer_task], verbose=True)
summarizer_result = await summarizer_crew.kickoff_async(inputs={"raw_article": full_text})
takeaways = parse_json_output(summarizer_result).get("takeaways", [])

final_output = {
    "status": "ready",
    "title": selected_meta["title"],
    "source": selected_meta["source"],
    "published_date": selected_meta["published_date"],
    "url": selected_meta["url"],
    "article": full_text,   # deterministic, verbatim — no LLM in this path
    "takeaways": takeaways,
}

print(json.dumps(final_output, indent=2))



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3b1581f9-e4e3-4eb3-9432-4bf5e34fb04f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      ARTICLE:                                                                                                   │
│      You don’t have any Active Subscription.                                                                    │
│                                                                                                                 │
│  Subscribed with another email? Logout and Login with that one.                                                 │
│                                                                                                                 │
│  Account subscription benefits alongside Premium Stories, Editorials,                                           │
│                Opinions and more. Unlock these with Subscription                                                │
│                                                                                                                 │
│  Need help with your subscription?                                                                              │
│                                                                                                                 │
│  Published - September 17, 2026 12:01 am IST                                                                    │
│                                                                                                                 │
│  Patterns such as repeat missing episodes, school discontinuity, parental loss or previous helpline contact     │
│  may help trigger support before vulnerability becomes a crisis.                                                │
│                                            | Photo Credit: File                                                 │
│                                                                                                                 │
│  The story of Saroo Brierley is an extraordinary one. Lost at the age of five after accidentally boarding a     │
│  train, separated from his family, and eventually adopted by an Australian couple, he returned to India more    │
│  than two decades later to find his biological mother. His journey, chronicled in A Long Way Home and later     │
│  adapted into the film Lion , is a story not only of resilience but also of identity, belonging, memory, and    │
│  the long consequences of childhood separation.                                                                 │
│                                                                                                                 │
│  As implementation of the Juvenile Justice (Care and Protection of Children) Act, 2015 has completed a decade,  │
│  India’s child-protection system is ready for its next transition: from collecting data about cases to          │
│  understanding whether or not state intervention actually changed a child’s life.                               │
│                                                                                                                 │
│  The first decade has created a substantial institutional and digital architecture. Child Welfare Committees,   │
│  Juvenile Justice Boards, District Child Protection Units, Special Juvenile Police Units, Child Care            │
│  Institutions, and helplines generate enormous amounts of information. Digital platforms increasingly capture   │
│  data on adoption, missing children, restoration, institutional and non-institutional care, sponsorship,        │
│  foster care, and aftercare.                           

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Summarizer                                                                                      │
│                                                                                                                 │
│  Task:                                                                                                          │
│      ARTICLE:                                                                                                   │
│      You don’t have any Active Subscription.                                                                    │
│                                                                                                                 │
│  Subscribed with another email? Logout and Login with that one.                                                 │
│                                                                                                                 │
│  Account subscription benefits alongside Premium Stories, Editorials,                                           │
│                Opinions and more. Unlock these with Subscription                                                │
│                                                                                                                 │
│  Need help with your subscription?                                                                              │
│                                                                                                                 │
│  Published - September 17, 2026 12:01 am IST                                                                    │
│                                                                                                                 │
│  Patterns such as repeat missing episodes, school discontinuity, parental loss or previous helpline contact     │
│  may help trigger support before vulnerability becomes a crisis.                                                │
│                                            | Photo Credit: File                                                 │
│                                                                                                                 │
│  The story of Saroo Brierley is an extraordinary one. Lost at the age of five after accidentally boarding a     │
│  train, separated from his family, and eventually adopted by an Australian couple, he returned to India more    │
│  than two decades later to find his biological mother. His journey, chronicled in A Long Way Home and later     │
│  adapted into the film Lion , is a story not only of resilience but also of identity, belonging, memory, and    │
│  the long consequences of childhood separation.                                                                 │
│                                                                                                                 │
│  As implementation of the Juvenile Justice (Care and Protection of Children) Act, 2015 has completed a decade,  │
│  India’s child-protection system is ready for its next transition: from collecting data about cases to          │
│  understanding whether or not state intervention actually changed a child’s life.                               │
│                                                                                                                 │
│  The first decade has created a substantial institutional and digital architecture. Child Welfare Committees,   │
│  Juvenile Justice Boards, District Child Protection Units, Special Juvenile Police Units, Child Care            │
│  Institutions, and helplines generate enormous amounts of information. Digital platforms increasingly capture   │
│  data on adoption, missing children, restoration, insti

time=2026-09-17T17:22:22.699Z level=INFO source=sched.go:1341 msg="updated VRAM based on existing loaded models" gpu=0 library=CUDA total="14.6 GiB" available="8.2 GiB"
time=2026-09-17T17:22:22.699Z level=INFO source=sched.go:1341 msg="updated VRAM based on existing loaded models" gpu=1 library=CUDA total="14.6 GiB" available="14.5 GiB"
time=2026-09-17T17:22:22.755Z level=INFO source=sched.go:1018 msg="selecting single GPU for llama-server model" main_gpu=0 id=1 filter_id=1 library=CUDA name=CUDA1 description="Tesla T4" integrated=false predicted="4.8 GiB" available="14.5 GiB"
time=2026-09-17T17:22:22.755Z level=INFO source=sched.go:1137 msg="selecting GPU backend for llama-server model" library=CUDA gpu_count=1 available_gpu_count=2
time=2026-09-17T17:22:22.755Z level=INFO source=sched.go:567 msg="llama-server model fits alongside existing models" predicted="5.5 GiB" predicted_num_ctx=32768 num_batch=1024 available="14.5 GiB" gpu_free="14.5 GiB" system_free="17.8 GiB" system_limited=f

[GIN] 2026/09/17 - 17:23:11 | 200 | 50.588978845s |       127.0.0.1 | POST     "/v1/chat/completions"
[Finalize] todos_count=0, todos_with_results=0


slot print_timing: id  0 | task 0 | prompt eval time =   12491.78 ms /  2170 tokens (    5.76 ms per token,   173.71 tokens per second)
slot print_timing: id  0 | task 0 |        eval time =    3346.97 ms /   377 tokens (    8.90 ms per token,   112.34 tokens per second)
slot print_timing: id  0 | task 0 |       total time =   15838.75 ms /  2547 tokens
slot print_timing: id  0 | task 0 |    graphs reused =        375
slot      release: id  0 | task 0 | stop processing: n_tokens = 2546, truncated = 0
srv  update_slots: all slots are idle


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Summarizer                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  { "takeaways": [ "The Juvenile Justice (Care and Protection of Children) Act, 2015 has completed a decade,     │
│  marking a significant milestone in India's child-protection system.", "Individual Care Plans (ICPs) must       │
│  evolve from procedural documents to dynamic, living tools that track a child's needs, interventions, and       │
│  outcomes in real time.", "Child-protection systems require simultaneous measurement of both the child's        │
│  well-being and the capacity of institutions to deliver care, ensuring accountability and addressing systemic   │
│  vulnerabilities." ] }                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      ARTICLE:                                                                                                   │
│      You don’t have any Active Subscription.                                                                    │
│                                                                                                                 │
│  Subscribed with another email? Logout and Login with that one.                                                 │
│                                                                                                                 │
│  Account subscription benefits alongside Premium Stories, Editorials,                                           │
│                Opinions and more. Unlock these with Subscription                                                │
│                                                                                                                 │
│  Need help with your subscription?                                                                              │
│                                                                                                                 │
│  Published - September 17, 2026 12:01 am IST                                                                    │
│                                                                                                                 │
│  Patterns such as repeat missing episodes, school discontinuity, parental loss or previous helpline contact     │
│  may help trigger support before vulnerability becomes a crisis.                                                │
│                                            | Photo Credit: File                                                 │
│                                                                                                                 │
│  The story of Saroo Brierley is an extraordinary one. Lost at the age of five after accidentally boarding a     │
│  train, separated from his family, and eventually adopted by an Australian couple, he returned to India more    │
│  than two decades later to find his biological mother. His journey, chronicled in A Long Way Home and later     │
│  adapted into the film Lion , is a story not only of resilience but also of identity, belonging, memory, and    │
│  the long consequences of childhood separation.                                                                 │
│                                                                                                                 │
│  As implementation of the Juvenile Justice (Care and Protection of Children) Act, 2015 has completed a decade,  │
│  India’s child-protection system is ready for its next transition: from collecting data about cases to          │
│  understanding whether or not state intervention actually changed a child’s life.                               │
│                                                                                                                 │
│  The first decade has created a substantial institutional and digital architecture. Child Welfare Committees,   │
│  Juvenile Justice Boards, District Child Protection Units, Special Juvenile Police Units, Child Care            │
│  Institutions, and helplines generate enormous amounts of information. Digital platforms increasingly capture   │
│  data on adoption, missing children, restoration, institutional and non-institutional care, sponsorship,        │
│  foster care, and aftercare.                           

{
  "status": "ready",
  "title": "Taking a holistic view of child protection",
  "source": "The Hindu (Op-Ed)",
  "published_date": "Wed, 16 Sep 2026 22:08:01 +0530",
  "url": "https://www.thehindu.com/opinion/op-ed/taking-a-holistic-view-of-child-protection/article71473865.ece",
  "article": "You don\u2019t have any Active Subscription.\n\nSubscribed with another email? Logout and Login with that one.\n\nAccount subscription benefits alongside Premium Stories, Editorials,\n              Opinions and more. Unlock these with Subscription\n\nNeed help with your subscription?\n\nPublished - September 17, 2026 12:01 am IST\n\nPatterns such as repeat missing episodes, school discontinuity, parental loss or previous helpline contact may help trigger support before vulnerability becomes a crisis.\u00a0\n                                          | Photo Credit: File\n\nThe story of Saroo Brierley is an extraordinary one. Lost at the age of five after accidentally boarding a train, separated f

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3b1581f9-e4e3-4eb3-9432-4bf5e34fb04f                                                                       │
│  Final Output: { "takeaways": [ "The Juvenile Justice (Care and Protection of Children) Act, 2015 has           │
│  completed a decade, marking a significant milestone in India's child-protection system.", "Individual Care     │
│  Plans (ICPs) must evolve from procedural documents to dynamic, living tools that track a child's needs,        │
│  interventions, and outcomes in real time.", "Child-protection systems require simultaneous measurement of      │
│  both the child's well-being and the capacity of institutions to deliver care, ensuring accountability and      │
│  addressing systemic vulnerabilities." ] }                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [19]:

# ============================================================
# CELL 8: vocab pick — two-task chain, explainer never sees the article
# Both agents run on the ONE already-selected article (no cross-comparison),
# so fast_llm is the right call per the earlier "compare vs. operate on one"
# rule — bump word_selector_agent to selector_llm only if it keeps picking
# bad words after this.
# ============================================================

word_selector_agent = Agent(
    role="Vocabulary Word Selector",
    goal="Pick one common, everyday English word from an article, ignoring the article's subject.",
    backstory=(
        "You scan articles purely to spot generally useful English words. "
        "You never pick technical, topic-specific, or jargon words."
    ),
    llm=fast_llm,
    verbose=True,
)

word_selector_task = Task(
    description="""
    Read the article below and pick ONE single English word from it.

    The word must be useful in everyday conversation, casual writing, or
    professional email — NOT useful only because of this article's topic.

    Reject: technical terms, AI/tech jargon, scientific jargon, business
    jargon, political terms, acronyms, names, places, organizations, phrases.

    Target register — common-but-slightly-elevated words a learner would
    genuinely want to add to their speech, e.g.: reluctant, meticulous,
    feasible, subtle, resilient, candid, versatile, ambiguous, tedious,
    plausible, discreet, thorough, pragmatic, inevitable, redundant.

    Self-check before answering: "Would this word be equally useful to
    someone who never reads this article and knows nothing about its topic?"
    If no, pick a different word.

    ARTICLE:
    {raw_article}
    """,
    expected_output="ONLY the single chosen word. No punctuation, no quotes, no explanation.",
    agent=word_selector_agent,
)

vocab_explainer_agent = Agent(
    role="English Vocabulary Coach",
    goal="Explain a single English word like a dictionary and a good teacher would, using only everyday-life examples.",
    backstory=(
        "You are a vocabulary coach who has never read any article. You "
        "only ever receive one isolated word and explain it using ordinary "
        "situations — school, work, family, friends, travel, money, hobbies."
    ),
    llm=fast_llm,
    verbose=True,
)

vocab_explain_task = Task(
    description="""
    You will receive exactly one English word from the previous task's
    output. You know NOTHING about where it came from — do not infer or
    reference any topic, subject, or article.

    Using ONLY that word, in plain general English, provide:
    - MEANING: the standard dictionary definition, in simple words
    - EXAMPLE: one sentence set in an ordinary daily-life situation
      (school, work, friends, family, travel, money, hobbies)
    - PRACTICE_SCENARIO: a short real situation where the reader could
      naturally use this word today

    If your answer mentions anything technical, AI-related, or subject-
    specific, you have failed — rewrite it in a plain daily-life context.
    """,
    expected_output="""
    Return ONLY valid JSON:
    {
      "word": "...",
      "meaning": "...",
      "example": "...",
      "practice_scenario": "..."
    }
    """,
    agent=vocab_explainer_agent,
    context=[word_selector_task],  # gets ONLY the word — {raw_article} is never in this task's description
)

vocab_crew = Crew(
    agents=[word_selector_agent, vocab_explainer_agent],
    tasks=[word_selector_task, vocab_explain_task],
    verbose=True,
)

vocab_result = await vocab_crew.kickoff_async(inputs={"raw_article": full_text})
vocab_word = parse_json_output(vocab_result)

final_output["vocab_word"] = vocab_word
print(json.dumps(final_output, indent=2))



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b16ea406-ee34-4c40-9993-779a2fe7207a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Read the article below and pick ONE single English word from it.                                           │
│                                                                                                                 │
│      The word must be useful in everyday conversation, casual writing, or                                       │
│      professional email — NOT useful only because of this article's topic.                                      │
│                                                                                                                 │
│      Reject: technical terms, AI/tech jargon, scientific jargon, business                                       │
│      jargon, political terms, acronyms, names, places, organizations, phrases.                                  │
│                                                                                                                 │
│      Target register — common-but-slightly-elevated words a learner would                                       │
│      genuinely want to add to their speech, e.g.: reluctant, meticulous,                                        │
│      feasible, subtle, resilient, candid, versatile, ambiguous, tedious,                                        │
│      plausible, discreet, thorough, pragmatic, inevitable, redundant.                                           │
│                                                                                                                 │
│      Self-check before answering: "Would this word be equally useful to                                         │
│      someone who never reads this article and knows nothing about its topic?"                                   │
│      If no, pick a different word.                                                                              │
│                                                                                                                 │
│      ARTICLE:                                                                                                   │
│      You don’t have any Active Subscription.                                                                    │
│                                                                                                                 │
│  Subscribed with another email? Logout and Login with that one.                                                 │
│                                                                                                                 │
│  Account subscription benefits alongside Premium Stories, Editorials,                                           │
│                Opinions and more. Unlock these with Subscription                                                │
│                                                                                                                 │
│  Need help with your subscription?                                                                              │
│                                                                                                                 │
│  Published - September 17, 2026 12:01 am IST                                                                    │
│                                                                                                                 │
│  Patterns such as repeat missing episodes, school discontinuity, parental loss or previous helpline contact     │
│  may help trigger support before vulnerability becomes 

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Vocabulary Word Selector                                                                                │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Read the article below and pick ONE single English word from it.                                           │
│                                                                                                                 │
│      The word must be useful in everyday conversation, casual writing, or                                       │
│      professional email — NOT useful only because of this article's topic.                                      │
│                                                                                                                 │
│      Reject: technical terms, AI/tech jargon, scientific jargon, business                                       │
│      jargon, political terms, acronyms, names, places, organizations, phrases.                                  │
│                                                                                                                 │
│      Target register — common-but-slightly-elevated words a learner would                                       │
│      genuinely want to add to their speech, e.g.: reluctant, meticulous,                                        │
│      feasible, subtle, resilient, candid, versatile, ambiguous, tedious,                                        │
│      plausible, discreet, thorough, pragmatic, inevitable, redundant.                                           │
│                                                                                                                 │
│      Self-check before answering: "Would this word be equally useful to                                         │
│      someone who never reads this article and knows nothing about its topic?"                                   │
│      If no, pick a different word.                                                                              │
│                                                                                                                 │
│      ARTICLE:                                                                                                   │
│      You don’t have any Active Subscription.                                                                    │
│                                                                                                                 │
│  Subscribed with another email? Logout and Login with that one.                                                 │
│                                                                                                                 │
│  Account subscription benefits alongside Premium Stories, Editorials,                                           │
│                Opinions and more. Unlock these with Subscription                                                │
│                                                                                                                 │
│  Need help with your subscription?                                                                              │
│                                                                                                                 │
│  Published - September 17, 2026 12:01 am IST                                                                    │
│                                                                                                                 │
│  Patterns such as repeat missing episodes, school disco

srv  server_strea: conv_id= (empty=1)
slot get_availabl: id  0 | task -1 |  - checking sim = 0.002 (5/2246) > 0.100
slot get_availabl: id  0 | task -1 | selected slot by LRU, t_last = 914482192
srv  get_availabl: updating prompt cache
srv   prompt_save:  - saving prompt with length 2546, total state size = 278.499 MiB (draft: 0.000 MiB)
srv          load:  - looking for better prompt, base f_keep = 0.002, f_sim = 0.002
srv          load:    - prompt with length    2546, lcp =       5, f_keep = 0.002, f_sim = 0.002
srv        update:  - cache state: 1 prompts, 278.499 MiB (limits: 8192.000 MiB, 32768 tokens, 74890 est)
srv        update:    - prompt 0x4ce2a9e0:    2546 tokens, checkpoints:  0,   278.499 MiB
srv  get_availabl: prompt cache update took 265.46 ms
slot launch_slot_: id  0 | task -1 | sampler chain: logits -> ?penalties -> ?dry -> ?top-n-sigma -> top-k -> ?typical -> ?top-p -> ?min-p -> ?xtc -> temp-ext -> dist 
slot launch_slot_: id  0 | task -1 | sampler params: 
	repeat_l

[GIN] 2026/09/17 - 17:33:11 | 500 |         10m0s |       127.0.0.1 | POST     "/v1/chat/completions"


...

slot print_timing: id  0 | task 36203 | prompt eval time =      73.15 ms /   319 tokens (    0.23 ms per token,  4361.08 tokens per second)
slot print_timing: id  0 | task 36203 |        eval time =    3619.95 ms /   421 tokens (    8.62 ms per token,   116.02 tokens per second)
slot print_timing: id  0 | task 36203 |       total time =    3693.10 ms /   740 tokens
slot print_timing: id  0 | task 36203 |    graphs reused =      36483
slot      release: id  0 | task 36203 | stop processing: n_tokens = 744, truncated = 0
srv  update_slots: all slots are idle


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: English Vocabulary Coach                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "word": "outcomes",                                                                                          │
│    "meaning": "The results or consequences of an event.",                                                       │
│    "example": "The teacher explained the students' grades as the outcomes of their efforts in the project.",    │
│    "practice_scenario": "A parent discusses their child's graduation as the outcome of years of hard work and   │
│  school."                                                                                                       │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      You will receive exactly one English word from the previous task's                                         │
│      output. You know NOTHING about where it came from — do not infer or                                        │
│      reference any topic, subject, or article.                                                                  │
│                                                                                                                 │
│      Using ONLY that word, in plain general English, provide:                                                   │
│      - MEANING: the standard dictionary definition, in simple words                                             │
│      - EXAMPLE: one sentence set in an ordinary daily-life situation                                            │
│        (school, work, friends, family, travel, money, hobbies)                                                  │
│      - PRACTICE_SCENARIO: a short real situation where the reader could                                         │
│        naturally use this word today                                                                            │
│                                                                                                                 │
│      If your answer mentions anything technical, AI-related, or subject-                                        │
│      specific, you have failed — rewrite it in a plain daily-life context.                                      │
│                                                                                                                 │
│  Agent: English Vocabulary Coach                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{
  "status": "ready",
  "title": "Taking a holistic view of child protection",
  "source": "The Hindu (Op-Ed)",
  "published_date": "Wed, 16 Sep 2026 22:08:01 +0530",
  "url": "https://www.thehindu.com/opinion/op-ed/taking-a-holistic-view-of-child-protection/article71473865.ece",
  "article": "You don\u2019t have any Active Subscription.\n\nSubscribed with another email? Logout and Login with that one.\n\nAccount subscription benefits alongside Premium Stories, Editorials,\n              Opinions and more. Unlock these with Subscription\n\nNeed help with your subscription?\n\nPublished - September 17, 2026 12:01 am IST\n\nPatterns such as repeat missing episodes, school discontinuity, parental loss or previous helpline contact may help trigger support before vulnerability becomes a crisis.\u00a0\n                                          | Photo Credit: File\n\nThe story of Saroo Brierley is an extraordinary one. Lost at the age of five after accidentally boarding a train, separated f

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b16ea406-ee34-4c40-9993-779a2fe7207a                                                                       │
│  Final Output: {                                                                                                │
│    "word": "outcomes",                                                                                          │
│    "meaning": "The results or consequences of an event.",                                                       │
│    "example": "The teacher explained the students' grades as the outcomes of their efforts in the project.",    │
│    "practice_scenario": "A parent discusses their child's graduation as the outcome of years of hard work and   │
│  school."                                                                                                       │
│  }                                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [20]:
# ============================================================
# CELL 9: HTML template — fixed, no AI agent
# ============================================================

from datetime import date
import html as html_lib

def build_html(data):
    def esc(s):
        return html_lib.escape(str(s)) if s else ""

    takeaways_html = "\n".join(
        f"<li>{esc(t)}</li>" for t in data.get("takeaways", [])
    )

    # article paragraphs -> <p> tags (article text uses \n\n between paragraphs)
    article_paragraphs = "\n".join(
        f"<p>{esc(para)}</p>" for para in data.get("article", "").split("\n\n") if para.strip()
    )

    vocab = data.get("vocab_word", {})
    today_str = date.today().strftime("%d %B %Y")

    html_doc = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Daily Brief AI — {esc(today_str)}</title>
<style>
    @page {{
        size: A4;
        margin: 2.2cm 2cm;
    }}
    body {{
        font-family: Georgia, 'Times New Roman', serif;
        color: #1a1a1a;
        line-height: 1.55;
        font-size: 12pt;
    }}
    .masthead {{
        text-align: center;
        border-bottom: 3px double #1a1a1a;
        padding-bottom: 10px;
        margin-bottom: 18px;
    }}
    .masthead .brand {{
        font-size: 13pt;
        letter-spacing: 3px;
        font-weight: bold;
        text-transform: uppercase;
    }}
    .masthead .date {{
        font-size: 10pt;
        color: #555;
        margin-top: 4px;
    }}
    h1.article-title {{
        font-size: 20pt;
        margin: 18px 0 4px 0;
        line-height: 1.25;
    }}
    .meta {{
        font-size: 9.5pt;
        color: #555;
        border-bottom: 1px solid #ccc;
        padding-bottom: 10px;
        margin-bottom: 16px;
    }}
    .divider {{
        border: none;
        border-top: 1px solid #999;
        margin: 22px 0;
    }}
    .section-label {{
        font-size: 11pt;
        letter-spacing: 2px;
        font-weight: bold;
        text-transform: uppercase;
        margin-bottom: 10px;
        color: #1a1a1a;
    }}
    .takeaways ul {{
        margin: 0;
        padding-left: 20px;
    }}
    .takeaways li {{
        margin-bottom: 6px;
    }}
    .vocab-box {{
        border: 1px solid #1a1a1a;
        padding: 16px 18px;
    }}
    .vocab-word {{
        font-size: 16pt;
        font-weight: bold;
        text-transform: uppercase;
        letter-spacing: 1px;
        margin-bottom: 10px;
    }}
    .vocab-field {{
        margin-bottom: 8px;
    }}
    .vocab-field .label {{
        font-weight: bold;
        font-size: 9.5pt;
        text-transform: uppercase;
        letter-spacing: 1px;
        color: #444;
        display: block;
        margin-bottom: 2px;
    }}
    .source-footer {{
        font-size: 9pt;
        color: #666;
        text-align: center;
        margin-top: 10px;
    }}
    .source-footer a {{
        color: #444;
    }}
    .vocab-box, .takeaways, .masthead {{
    page-break-inside: avoid;
    }}

    h1.article-title {{
    page-break-after: avoid;
    }}
    .meta {{
    page-break-before: avoid;
    }}

    p {{
    orphans: 3;
    widows: 3;
    }}

</style>
</head>
<body>

<div class="masthead">
    <div class="brand">Daily Brief AI</div>
    <div class="date">{esc(today_str)}</div>
</div>

<h1 class="article-title">{esc(data.get('title', ''))}</h1>
<div class="meta">{esc(data.get('source', 'Unknown source'))} &nbsp;|&nbsp; {esc(data.get('published_date', ''))}</div>

{article_paragraphs}

<hr class="divider">

<div class="takeaways">
    <div class="section-label">Key Takeaways</div>
    <ul>
    {takeaways_html}
    </ul>
</div>

<hr class="divider">

<div class="vocab-box">
    <div class="section-label">Word of the Day</div>
    <div class="vocab-word">{esc(vocab.get('word', ''))}</div>
    <div class="vocab-field">
        <span class="label">Meaning</span>
        {esc(vocab.get('meaning', ''))}
    </div>
    <div class="vocab-field">
        <span class="label">Example</span>
        {esc(vocab.get('example', ''))}
    </div>
    <div class="vocab-field">
        <span class="label">Practice</span>
        {esc(vocab.get('practice_scenario', ''))}
    </div>
</div>

<hr class="divider">

<div class="source-footer">
    Original source: <a href="{esc(data.get('url', ''))}">{esc(data.get('url', ''))}</a>
</div>

</body>
</html>"""
    return html_doc


html_content = build_html(final_output)

html_path = f"/kaggle/working/daily_brief_{date.today()}.html"
with open(html_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print("HTML written to:", html_path)

HTML written to: /kaggle/working/daily_brief_2026-09-17.html


In [21]:
# ============================================================
# CELL 10: HTML -> PDF
# Run the apt install once per Colab session (before first use).
# ============================================================

!apt-get -qq install -y libpango-1.0-0 libpangocairo-1.0-0 libgdk-pixbuf2.0-0 libcairo2 > /dev/null
!pip -q install weasyprint

from weasyprint import HTML

pdf_path = f"/kaggle/working/daily_brief_{date.today()}.pdf"
HTML(string=html_content).write_pdf(pdf_path)

print("PDF written to:", pdf_path)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.9/328.9 kB 5.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 34.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.4/829.4 kB 43.7 MB/s eta 0:00:00


PDF written to: /kaggle/working/daily_brief_2026-09-17.pdf


In [22]:
# ============================================================
# CELL 11: Email — send the PDF
# ============================================================

import smtplib
from email.message import EmailMessage
# _secrets already created in Cell 1 (kaggle_secrets.UserSecretsClient) — reused here
GMAIL_ADDRESS = _secrets.get_secret('GMAIL_ADDRESS')       # your sending address
GMAIL_APP_PASSWORD = _secrets.get_secret('GMAIL_APP_PASSWORD')
RECIPIENT_EMAIL = _secrets.get_secret('RECIPIENT_EMAIL')   # can be same as GMAIL_ADDRESS

def send_daily_brief_email(pdf_path, article_title):
    msg = EmailMessage()
    msg["Subject"] = f"Daily Brief AI — {date.today().strftime('%d %B %Y')}"
    msg["From"] = GMAIL_ADDRESS
    msg["To"] = RECIPIENT_EMAIL
    msg.set_content(
        f"Your daily reading brief is ready.\n\n"
        f"Today's article: {article_title}\n\n"
        f"See attached PDF."
    )

    with open(pdf_path, "rb") as f:
        pdf_data = f.read()
    msg.add_attachment(
        pdf_data,
        maintype="application",
        subtype="pdf",
        filename=os.path.basename(pdf_path),
    )

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
        smtp.login(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)
        smtp.send_message(msg)

    print("Email sent to", RECIPIENT_EMAIL)


send_daily_brief_email(pdf_path, final_output["title"])

Email sent to work.maanas@gmail.com


In [23]:
# Sanity check — confirm the full pipeline actually completed
print("Pipeline finished. PDF:", pdf_path)
print("Title:", final_output["title"])


Pipeline finished. PDF: /kaggle/working/daily_brief_2026-09-17.pdf
Title: Taking a holistic view of child protection
